# Using notebooks for inspection

In this example, we show how the results of a previous run can be loaded in a Jupyter notebook for further inspection (including any intermediate results).

To start with, we run the segmentation example pipeline.

In [1]:
!command rm -f $(find examples -name 'data.dill.gz')
!command rm -f $(find examples -name '.task.json')
!command rm -f $(find examples -name '.sha.json')

find: examples: No such file or directory
find: examples: No such file or directory
find: examples: No such file or directory


In [2]:
%cd ..

from IPython.display import Image, display

import os
import pathlib
import shutil
import tempfile

rootdir = pathlib.Path('.').resolve()
tempdir = tempfile.TemporaryDirectory()
tempdir_path = pathlib.Path(tempdir.name)
task_root_path = tempdir_path / 'examples' / 'segmentation'
shutil.copytree('examples/segmentation', task_root_path)
os.symlink(rootdir / 'repype', tempdir_path / 'repype', target_is_directory = True)
os.symlink(rootdir / 'tests', tempdir_path / 'tests', target_is_directory = True)
os.chdir(tempdir_path)

/Users/void/Dev/repype


/Users/void/Dev/repype/env/lib/python3.11/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


## Creating some results to work with

We first create some results to work with. We only run the task specified in `examples/segmentation/task.yml` and omit the `sigma=2` variant:

In [3]:
!command python -m repype examples/segmentation --task examples/segmentation --run


1 task(s) selected for running
  
  (1/1) Entering task: /private/var/folders/hs/cqtqb2dn29d5nfz8d_qm6f6w0000gn/T/tmp8zxi13z0/examples/segmentation
  Starting from scratch
    
    (1/1) Processing: B2--W00026--P00001--Z00000--T00000--dapi.tif
                               
  Results have been stored ✅


# Loading the results for inspection

We load the results that were written by the previous run of the task:

In [4]:
import repype.batch

batch = repype.batch.Batch()
batch.load('examples')
batch.tasks

{PosixPath('examples/segmentation'): <Task "examples/segmentation" 5bde8ec>,
 PosixPath('examples/segmentation/sigma=2'): <Task "examples/segmentation/sigma=2" 678f085>}

In [5]:
task = batch.task('examples/segmentation')
task_data = task.load()
task_data.keys()

dict_keys(['B2--W00026--P00001--Z00000--T00000--dapi.tif'])

In [6]:
pipeline_data = task_data['B2--W00026--P00001--Z00000--T00000--dapi.tif']
pipeline_data.keys()

dict_keys(['input_id', 'image', 'segmentation'])

In [7]:
pipeline_data['segmentation']

array([[255, 255, 255, ...,   0,   0,   0],
       [  0,   0,   0, ...,   0,   0,   0],
       [  0,   0,   0, ...,   0,   0,   0],
       ...,
       [  0,   0,   0, ...,   0,   0,   0],
       [  0,   0,   0, ...,   0,   0,   0],
       [  0,   0,   0, ...,   0,   0,   0]],
      shape=(1024, 1344), dtype=uint8)

# Re-computing marginal data

Data that is marginal will be missing when loaded as shown above. To inspect marginal data, it is necessary to run the corresponding computations:

In [ ]:
config = task.create_config()
task_data = task.run(config=config)

{'B2--W00026--P00001--Z00000--T00000--dapi.tif': {'input_id': 'B2--W00026--P00001--Z00000--T00000--dapi.tif',
  'image': array([[33372, 33358, 33350, ..., 33092, 33084, 33089],
         [33315, 33316, 33322, ..., 33076, 33086, 33087],
         [33221, 33222, 33245, ..., 33086, 33069, 33074],
         ...,
         [33187, 33170, 33166, ..., 33114, 33127, 33131],
         [33198, 33176, 33170, ..., 33129, 33132, 33129],
         [33223, 33191, 33191, ..., 33122, 33120, 33128]],
        shape=(1024, 1344), dtype=uint16),
  'segmentation': array([[255, 255, 255, ...,   0,   0,   0],
         [  0,   0,   0, ...,   0,   0,   0],
         [  0,   0,   0, ...,   0,   0,   0],
         ...,
         [  0,   0,   0, ...,   0,   0,   0],
         [  0,   0,   0, ...,   0,   0,   0],
         [  0,   0,   0, ...,   0,   0,   0]],
        shape=(1024, 1344), dtype=uint8)}}